In [7]:
# ============================================================
# 010 — UI
# ============================================================

from google.colab import drive
drive.mount("/content/drive")


from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML


# ============================================================
# PROJECT CONFIG
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/FitnessML_Master"
)

RESULTS_DIR = PROJECT_ROOT / "results"
FINAL_DIR = RESULTS_DIR / "009_final_results"


# ============================================================
# FINAL ARTIFACTS
# ============================================================

MODEL_PATH = FINAL_DIR / "random_forest_final.pkl"
PREDICTIONS_PATH = FINAL_DIR / "prediction_results.csv"
MODELING_PATH = FINAL_DIR / "fitbit_modeling_features.csv"
METADATA_PATH = FINAL_DIR / "model_metadata.json"


print("=" * 70)
print("010 — UI")
print("=" * 70)

print(f"Project root: {PROJECT_ROOT}")
print(f"Final results: {FINAL_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
010 — UI
Project root: /content/drive/MyDrive/FitnessML_Master
Final results: /content/drive/MyDrive/FitnessML_Master/results/009_final_results


In [8]:
# ============================================================
# CHECK FINAL ARTIFACTS
# ============================================================

required_files = [
    MODEL_PATH,
    PREDICTIONS_PATH,
    MODELING_PATH,
    METADATA_PATH,
]

for path in required_files:

    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{path}"
        )

    print(f"✓ {path.name}")


print("=" * 70)
print("FINAL ARTIFACTS READY")
print("=" * 70)

✓ random_forest_final.pkl
✓ prediction_results.csv
✓ fitbit_modeling_features.csv
✓ model_metadata.json
FINAL ARTIFACTS READY


In [9]:
# ============================================================
# LOAD FINAL ARTIFACTS
# ============================================================

# ------------------------------------------------------------
# LOAD MODEL
# ------------------------------------------------------------

rf_model = joblib.load(MODEL_PATH)

print("✓ Random Forest model loaded")


# ------------------------------------------------------------
# LOAD MODELING DATA
# ------------------------------------------------------------

modeling = pd.read_csv(MODELING_PATH)

modeling["ActivityDate"] = pd.to_datetime(
    modeling["ActivityDate"]
)

modeling = (
    modeling
    .sort_values(
        ["Id", "ActivityDate"]
    )
    .reset_index(drop=True)
)

print(
    f"✓ Modeling data loaded: "
    f"{modeling.shape}"
)


# ------------------------------------------------------------
# LOAD PREDICTION RESULTS
# ------------------------------------------------------------

predictions_df = pd.read_csv(
    PREDICTIONS_PATH
)

predictions_df["ActivityDate"] = pd.to_datetime(
    predictions_df["ActivityDate"]
)

predictions_df = (
    predictions_df
    .sort_values(
        ["Id", "ActivityDate"]
    )
    .reset_index(drop=True)
)

print(
    f"✓ Prediction results loaded: "
    f"{predictions_df.shape}"
)


# ------------------------------------------------------------
# LOAD MODEL METADATA
# ------------------------------------------------------------

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8"
) as f:
    model_metadata = json.load(f)

print("✓ Model metadata loaded")


# ============================================================
# BASIC INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("MODEL / DATA READY")
print("=" * 70)

print(
    f"Model:       "
    f"{model_metadata.get('model', 'Unknown')}"
)

print(
    f"Target:      "
    f"{model_metadata.get('target', 'Unknown')}"
)

print(
    f"Modeling:    "
    f"{modeling.shape}"
)

print(
    f"Predictions: "
    f"{predictions_df.shape}"
)

print("=" * 70)

✓ Random Forest model loaded
✓ Modeling data loaded: (680, 68)
✓ Prediction results loaded: (125, 6)
✓ Model metadata loaded

MODEL / DATA READY
Model:       Random Forest — All Features
Target:      target_calories_next_day
Modeling:    (680, 68)
Predictions: (125, 6)


In [10]:
# ============================================================
# MODEL FEATURES / CONSISTENCY CHECK
# ============================================================

# ------------------------------------------------------------
# GET FEATURES EXPECTED BY THE MODEL
# ------------------------------------------------------------

if not hasattr(rf_model, "feature_names_in_"):
    raise AttributeError(
        "Фінальна Random Forest модель "
        "не містить feature_names_in_."
    )

feature_columns = list(
    rf_model.feature_names_in_
)


# ------------------------------------------------------------
# CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "Id",
    "ActivityDate",
    "target_calories_next_day",
]

missing_required = [
    col
    for col in required_columns
    if col not in modeling.columns
]

if missing_required:
    raise ValueError(
        "У modeling dataset відсутні службові/target колонки:\n"
        + "\n".join(missing_required)
    )


# ------------------------------------------------------------
# CHECK MODEL FEATURES
# ------------------------------------------------------------

missing_features = [
    col
    for col in feature_columns
    if col not in modeling.columns
]

extra_feature_columns = [
    col
    for col in modeling.columns
    if col not in feature_columns
    and col not in required_columns
]


if missing_features:
    raise ValueError(
        "У modeling dataset відсутні ознаки, "
        "необхідні фінальній моделі:\n"
        + "\n".join(missing_features)
    )


# ------------------------------------------------------------
# FEATURE SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("MODEL FEATURES / CONSISTENCY CHECK")
print("=" * 70)

print(
    f"Model features:        {len(feature_columns)}"
)

print(
    f"Modeling columns:      {len(modeling.columns)}"
)

print(
    f"Missing features:      {len(missing_features)}"
)

print(
    f"Extra feature columns: {len(extra_feature_columns)}"
)


print("\nFeature groups:")

base_features = [
    col
    for col in feature_columns
    if "_lag_" not in col
    and "rolling_" not in col
]

lag_features = [
    col
    for col in feature_columns
    if "_lag_" in col
]

rolling_features = [
    col
    for col in feature_columns
    if "rolling_" in col
]

print(f"  Base:      {len(base_features)}")
print(f"  Lag:       {len(lag_features)}")
print(f"  Rolling:   {len(rolling_features)}")


print("\nFirst features:")

for feature in feature_columns[:10]:
    print(f"  ✓ {feature}")

if len(feature_columns) > 10:
    print(
        f"  ... and {len(feature_columns) - 10} more"
    )


print("=" * 70)
print("FEATURE CHECK PASSED")
print("=" * 70)

MODEL FEATURES / CONSISTENCY CHECK
Model features:        65
Modeling columns:      68
Missing features:      0
Extra feature columns: 0

Feature groups:
  Base:      13
  Lag:       28
  Rolling:   24

First features:
  ✓ TotalSteps
  ✓ TotalDistance
  ✓ TrackerDistance
  ✓ LoggedActivitiesDistance
  ✓ VeryActiveDistance
  ✓ ModeratelyActiveDistance
  ✓ LightActiveDistance
  ✓ SedentaryActiveDistance
  ✓ VeryActiveMinutes
  ✓ FairlyActiveMinutes
  ... and 55 more
FEATURE CHECK PASSED


In [11]:
# ============================================================
# PREPARE DEMO DATA
# ============================================================

# ------------------------------------------------------------
# TEST OBSERVATIONS AVAILABLE FOR DEMONSTRATION
# ------------------------------------------------------------

demo_keys = (
    predictions_df[
        ["Id", "ActivityDate"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Id", "ActivityDate"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# AVAILABLE USERS
# ------------------------------------------------------------

user_ids = sorted(
    demo_keys["Id"].unique()
)


if len(user_ids) == 0:
    raise ValueError(
        "У prediction_results.csv "
        "не знайдено жодного користувача."
    )


# ------------------------------------------------------------
# BASIC VALIDATION
# ------------------------------------------------------------

demo_pairs = set(
    zip(
        demo_keys["Id"],
        demo_keys["ActivityDate"]
    )
)

modeling_pairs = set(
    zip(
        modeling["Id"],
        modeling["ActivityDate"]
    )
)

missing_demo_rows = [
    pair
    for pair in demo_pairs
    if pair not in modeling_pairs
]


if missing_demo_rows:
    raise ValueError(
        "Для частини тестових спостережень "
        "відсутні відповідні рядки в modeling dataset."
    )


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("DEMO DATA READY")
print("=" * 70)

print(
    f"Users available:       {len(user_ids)}"
)

print(
    f"Test observations:     {len(demo_keys)}"
)

print(
    f"First user:             {user_ids[0]}"
)

print(
    f"Date range:             "
    f"{demo_keys['ActivityDate'].min().date()} "
    f"→ "
    f"{demo_keys['ActivityDate'].max().date()}"
)

print("=" * 70)

DEMO DATA READY
Users available:       27
Test observations:     125
First user:             1503960366
Date range:             2016-05-07 → 2016-05-11


In [19]:
# ============================================================
# UI STYLE
# ============================================================

display(HTML("""
<style>

.fitness-card {
    max-width: 900px;
    margin: 25px auto;
    padding: 30px 34px;
    border: 1px solid #d6d9de;
    border-radius: 16px;
    background: #ffffff;
    box-shadow: 0 4px 18px rgba(0, 0, 0, 0.08);
}

.fitness-title {
    font-size: 28px;
    font-weight: 700;
    margin-bottom: 6px;
}

.fitness-subtitle {
    font-size: 15px;
    margin-bottom: 28px;
}

.fitness-section {
    font-size: 18px;
    font-weight: 600;
    margin: 20px 0 12px 0;
}

.fitness-label {
    font-size: 14px;
    font-weight: 600;
    margin-bottom: 5px;
}

.fitness-result {
    margin-top: 25px;
    padding: 22px;
    border-radius: 12px;
    border: 1px solid #d6d9de;
    background: #fafafa;
}

.fitness-metric {
    display: inline-block;
    vertical-align: top;
    width: 28%;
    min-width: 180px;
    margin: 6px;
    padding: 16px;
    border-radius: 10px;
    background: #f1f3f5;
}

.fitness-metric-label {
    font-size: 12px;
    margin-bottom: 7px;
}

.fitness-metric-value {
    font-size: 22px;
    font-weight: 700;
}

.fitness-note {
    margin-top: 14px;
    font-size: 13px;
}

</style>
"""))

print("✓ UI style loaded")

✓ UI style loaded


In [24]:
# ============================================================
# UI HELPER FUNCTIONS
# ============================================================

def get_dates_for_user(user_id):
    """
    Return dates available for the selected user
    in the test/prediction dataset.
    """

    return (
        demo_keys.loc[
            demo_keys["Id"] == user_id,
            "ActivityDate"
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )


def get_modeling_row(user_id, activity_date):
    """
    Return the modeling row for the selected
    user and activity date.
    """

    rows = modeling.loc[
        (modeling["Id"] == user_id)
        & (
            modeling["ActivityDate"]
            == pd.Timestamp(activity_date)
        )
    ]

    if rows.empty:
        raise ValueError(
            "Не знайдено відповідний рядок "
            "у modeling dataset."
        )

    return rows.iloc[0]


def get_prediction_row(user_id, activity_date):
    """
    Return the stored prediction result for
    the selected test observation.
    """

    rows = predictions_df.loc[
        (predictions_df["Id"] == user_id)
        & (
            predictions_df["ActivityDate"]
            == pd.Timestamp(activity_date)
        )
    ]

    if rows.empty:
        raise ValueError(
            "Не знайдено відповідний рядок "
            "у prediction results."
        )

    return rows.iloc[0]


def make_feature_vector(modeling_row):
    """
    Build the feature vector in exactly the order
    expected by the trained Random Forest model.
    """

    missing = [
        feature
        for feature in feature_columns
        if feature not in modeling_row.index
    ]

    if missing:
        raise ValueError(
            "У рядку відсутні ознаки:\n"
            + "\n".join(missing)
        )

    return pd.DataFrame(
        [[
            modeling_row[feature]
            for feature in feature_columns
        ]],
        columns=feature_columns,
    )


def predict_for_observation(user_id, activity_date):
    """
    Generate a prediction for one selected
    test observation.
    """

    modeling_row = get_modeling_row(
        user_id,
        activity_date
    )

    prediction_row = get_prediction_row(
        user_id,
        activity_date
    )

    X = make_feature_vector(
        modeling_row
    )

    predicted_calories = float(
        rf_model.predict(X)[0]
    )

    actual_calories = float(
        prediction_row["ActualCalories"]
    )

    absolute_error = abs(
        predicted_calories
        - actual_calories
    )

    return {
        "predicted": predicted_calories,
        "actual": actual_calories,
        "error": absolute_error,
        "modeling_row": modeling_row,
        "prediction_row": prediction_row,
    }


print("=" * 70)
print("UI HELPER FUNCTIONS READY")
print("=" * 70)

print("✓ User/date lookup")
print("✓ Modeling row lookup")
print("✓ Prediction row lookup")
print("✓ Feature vector construction")
print("✓ Model prediction")

UI HELPER FUNCTIONS READY
✓ User/date lookup
✓ Modeling row lookup
✓ Prediction row lookup
✓ Feature vector construction
✓ Model prediction


In [25]:
# ============================================================
# UI WIDGETS
# ============================================================

# ------------------------------------------------------------
# USER SELECTOR
# ------------------------------------------------------------

user_selector = widgets.Dropdown(
    options=user_ids,
    value=user_ids[0],
    description="User:",
    layout=widgets.Layout(width="420px"),
)


# ------------------------------------------------------------
# DATE SELECTOR
# ------------------------------------------------------------

initial_dates = get_dates_for_user(
    user_ids[0]
)

if not initial_dates:
    raise ValueError(
        "Для першого користувача "
        "не знайдено доступних дат."
    )


date_selector = widgets.Dropdown(
    options=initial_dates,
    value=initial_dates[0],
    description="Date:",
    layout=widgets.Layout(width="420px"),
)


# ------------------------------------------------------------
# PREDICTION BUTTON
# ------------------------------------------------------------

predict_button = widgets.Button(
    description="Calculate prediction",
    button_style="primary",
    icon="line-chart",
    layout=widgets.Layout(
        width="220px",
        height="40px"
    ),
)


# ------------------------------------------------------------
# OUTPUT AREA
# ------------------------------------------------------------

output_area = widgets.Output()


print("=" * 70)
print("UI WIDGETS READY")
print("=" * 70)

print("✓ User selector")
print("✓ Date selector")
print("✓ Prediction button")
print("✓ Output area")

UI WIDGETS READY
✓ User selector
✓ Date selector
✓ Prediction button
✓ Output area


In [26]:
# ============================================================
# USER → DATE CALLBACK
# ============================================================

def update_date_options(change):
    """
    Update available dates when the selected user changes.
    """

    if change["name"] != "value":
        return

    selected_user = change["new"]

    dates = get_dates_for_user(
        selected_user
    )

    if not dates:
        date_selector.options = []
        date_selector.value = None
        return

    date_selector.options = dates
    date_selector.value = dates[0]


user_selector.observe(
    update_date_options,
    names="value"
)


print("=" * 70)
print("USER → DATE CALLBACK READY")
print("=" * 70)

print(
    "✓ Changing user updates available dates"
)

USER → DATE CALLBACK READY
✓ Changing user updates available dates


In [30]:
# ============================================================
# RENDER PREDICTION RESULT
# ============================================================

def render_prediction(user_id, activity_date):
    """
    Build and display the prediction result
    for the selected user and date.
    """

    result = predict_for_observation(
        user_id,
        activity_date
    )

    predicted = result["predicted"]
    actual = result["actual"]
    error = result["error"]

    modeling_row = result["modeling_row"]


    # --------------------------------------------------------
    # RESULT HEADER
    # --------------------------------------------------------

    display(HTML(
        f"""
        <div class="ui-result">

            <div class="ui-section">
                Prediction result
            </div>

            <div class="ui-note">
                User: <b>{user_id}</b><br>
                Activity date:
                <b>{pd.Timestamp(activity_date).date()}</b>
            </div>

            <br>

            <div class="ui-metric">
                <div class="ui-metric-label">
                    Predicted calories
                </div>
                <div class="ui-metric-value">
                    {predicted:,.0f} kcal
                </div>
            </div>

            <div class="ui-metric">
                <div class="ui-metric-label">
                    Actual next-day calories
                </div>
                <div class="ui-metric-value">
                    {actual:,.0f} kcal
                </div>
            </div>

            <div class="ui-metric">
                <div class="ui-metric-label">
                    Absolute error
                </div>
                <div class="ui-metric-value">
                    {error:,.0f} kcal
                </div>
            </div>

        </div>
        """
    ))


    # --------------------------------------------------------
    # HISTORY
    # --------------------------------------------------------

    history = (
        modeling[
            modeling["Id"] == user_id
        ]
        .loc[
            lambda df:
            df["ActivityDate"]
            <= pd.Timestamp(activity_date)
        ]
        .sort_values("ActivityDate")
        .tail(14)
        .copy()
    )


    if history.empty:
        return


    # --------------------------------------------------------
    # HISTORY PLOT
    # --------------------------------------------------------

    plt.figure(figsize=(10, 4.5))

    plt.plot(
        history["ActivityDate"],
        history["Calories"],
        marker="o",
        label="Daily calories"
    )

    plt.axvline(
        pd.Timestamp(activity_date),
        linestyle="--",
        label="Selected date"
    )

    plt.xlabel("Activity date")
    plt.ylabel("Calories (kcal)")
    plt.title(
        f"Recent activity history — User {user_id}"
    )

    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [31]:
# ============================================================
# PREDICTION BUTTON CALLBACK
# ============================================================

def on_predict_button_click(button):
    """
    Handle prediction button click.
    """

    with output_area:
        output_area.clear_output()

        try:
            render_prediction(
                user_selector.value,
                date_selector.value
            )

        except Exception as error:
            display(
                HTML(
                    f"""
                    <div class="ui-result">
                        <b>Prediction error</b><br>
                        {error}
                    </div>
                    """
                )
            )


predict_button.on_click(
    on_predict_button_click
)


print("=" * 70)
print("PREDICTION CALLBACK READY")
print("=" * 70)

print(
    "✓ Prediction button connected"
)

PREDICTION CALLBACK READY
✓ Prediction button connected


In [32]:
# ============================================================
# DISPLAY UI
# ============================================================

header = widgets.HTML(
    """
    <div class="fitness-card">

        <div class="fitness-title">
            Fitbit Energy Expenditure Prediction
        </div>

        <div class="fitness-subtitle">
            Інтелектуальна система прогнозування
            добових енергетичних витрат
        </div>

        <div class="fitness-section">
            Вибір спостереження
        </div>

    </div>
    """
)


user_label = widgets.HTML(
    "<div class='fitness-label'>Користувач</div>"
)

date_label = widgets.HTML(
    "<div class='fitness-label'>Дата активності</div>"
)


controls = widgets.VBox(
    [
        user_label,
        user_selector,
        date_label,
        date_selector,
        widgets.HTML("<br>"),
        predict_button,
        output_area,
    ],
    layout=widgets.Layout(
        width="850px",
        margin="0 auto",
    )
)


display(
    widgets.VBox(
        [
            header,
            controls,
        ],
        layout=widgets.Layout(
            width="100%"
        )
    )
)

print("✓ UI displayed")

✓ UI displayed
